# Evaluation

In [ ]:
import sys
sys.path.append('../src')
from llama_cpp import Llama
from prompts import build_prompt

llm = Llama(model_path='../models/model-q4_k_m.gguf', n_ctx=1024, n_threads=8, verbose=False)

## Tester sur un exemple

In [ ]:
schema = 'CREATE TABLE employes (nom VARCHAR, poste VARCHAR, salaire INTEGER, ville VARCHAR)'
question = 'Quel est le salaire moyen par ville ?'

out = llm(build_prompt(question, schema), max_tokens=200, stop=['<|im_end|>'], temperature=0.0)
print(out['choices'][0]['text'].strip())

## Évaluation sur le set de validation (modèle fine-tuné)

Métriques : exact match normalisé + validité SQL (EXPLAIN sur SQLite en mémoire).

In [ ]:
!python ../src/evaluate.py --model ../models/model-q4_k_m.gguf --val_file ../data/processed/val.jsonl --out ../docs/results_finetuned.json

## Baseline : modèle de base non fine-tuné

Convertir le modèle de base brut en GGUF (même procédure, sans adapters) puis :

In [ ]:
!python ../src/evaluate.py --model ../models/base-model-q4_k_m.gguf --val_file ../data/processed/val.jsonl --out ../docs/results_base.json

In [ ]:
import json, pandas as pd
rows = []
for name, path in [('Base', '../docs/results_base.json'), ('Fine-tuné LoRA', '../docs/results_finetuned.json')]:
    r = json.load(open(path))
    rows.append({'Modèle': name, 'Exact match %': r['exact_match_pct'], 'SQL valide %': r['sql_valide_pct']})
pd.DataFrame(rows)